# 24 Eval Audio Heads

In [2]:
import numpy as np
import pandas as pd

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import f1_score, log_loss

from woofdecoder.config import REPO_ROOT
from wooftrainer.trainers.audio_heads import save_audio_heads, train_audio_heads

from woofdecoder.branches.audio import emotion_inputs, select_layer
from wooftrainer.trainers.embeddings import attach_labels, load_embeddings

## Evaluation Preparation

In [3]:
ENCODER = "beats_iter3_plus_as2m"
LAYER = "mean" # notebook 20
HEADS_DIR = REPO_ROOT/"models"/"audio"/ENCODER

In [4]:
heads = train_audio_heads(ENCODER, LAYER)
print(heads.summary.round(5).to_string(index=False))
print("\nnudge used:", heads.summary.loc[2, "l2"] != "none")

     head      C   l2  temperature  before_log_loss  before_macro_f1  after_log_loss  after_macro_f1
  arousal 0.0010           0.85746          0.89024          0.57157         0.88714         0.57157
  valence 0.0003           0.83079          1.03770          0.46126         1.03605         0.46126
situation 0.0003 none      0.78589          1.15417          0.51874         1.14319         0.51874

nudge used: False


## Helper Functions

In [5]:
def situation_probs(features):
    logits = heads.situation.logits(features)
    shift = heads.nudge.shift(emotion_inputs(features, heads.arousal, heads.valence))
    return heads.situation.probs_from_logits(logits + shift)

def bootstrap_ci(labels, predicted, repeats=1000, seed=0):
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(repeats):
        index = rng.integers(0, len(labels), len(labels))
        scores.append(f1_score(labels[index], predicted[index], average="macro"))
    return np.percentile(scores, [2.5, 97.5])

## Evaluation on Situation Head

In [6]:
x, table = load_embeddings("audio_context", "val", ENCODER)
table = attach_labels(table, "audio_context", "val")
probs = situation_probs(select_layer(x, LAYER))
classes = heads.situation.classes
table["predicted"] = np.array(classes)[probs.argmax(axis=1)]
 
for source, category in (("playpant", "playful_excited"), ("self_labeled", "distressed_lonely")):
    part = (table["source"] == source).to_numpy()
    recall = (table.loc[part, "predicted"] == category).mean()
    mean_p = probs[part, classes.index(category)].mean()
    print(f"\n{source}: {part.sum()} windows | recall {recall:.2f} | mean p({category}) {mean_p:.2f}")
    print(table.loc[part, "predicted"].value_counts(normalize=True).round(2).to_string())
 
# Self-labeled val per video (what the app would say for each whole video).
mine = (table["source"] == "self_labeled").to_numpy()
video_probs = pd.DataFrame(probs[mine], columns=classes)
video_probs["video"] = table.loc[mine, "clip_id"].str.split("@").str[0].to_numpy()
per_video = video_probs.groupby("video").mean()
per_video["predicted"] = per_video[classes].idxmax(axis=1)
print("\nself-labeled val, per video:")
print(per_video.round(2).to_string())


playpant: 403 windows | recall 0.21 | mean p(playful_excited) 0.22
predicted
distressed_lonely    0.62
playful_excited      0.21
other                0.18

self_labeled: 13 windows | recall 0.92 | mean p(distressed_lonely) 0.73
predicted
distressed_lonely    0.92
alert_aggressive     0.08

self-labeled val, per video:
                                              alert_aggressive  attention_seeking  distressed_lonely  other  playful_excited          predicted
video                                                                                                                                          
self_labeled/audio_distressedlonely/A002.mp4              0.04               0.06               0.68   0.20             0.02  distressed_lonely
self_labeled/audio_distressedlonely/A010.mp4              0.03               0.03               0.84   0.05             0.05  distressed_lonely
self_labeled/audio_distressedlonely/A011.mp4              0.05               0.04               0.78   

## Evaluation on Emotion Heads (EmotionalCanines Val)

In [8]:
ex, etable = load_embeddings("audio_emotion", "val", ENCODER)
etable = attach_labels(etable, "audio_emotion", "val")
features = select_layer(ex, LAYER)
 
for target, head in (("arousal", heads.arousal), ("valence", heads.valence)):
    labels = etable[target].to_numpy()
    p = head.predict_proba(features)
    predicted = np.array(head.classes)[p.argmax(axis=1)]
    low, high = bootstrap_ci(labels, predicted)
    print(f"{target}: {len(labels)} clips | macro-F1 {f1_score(labels, predicted, average='macro'):.3f} ")
    print(f"[{low:.3f}, {high:.3f}] | log-loss {log_loss(labels, p, labels=head.classes):.3f}")

arousal: 200 clips | macro-F1 0.500 
[0.431, 0.565] | log-loss 0.996
valence: 200 clips | macro-F1 0.384 
[0.314, 0.450] | log-loss 1.070
